# 16.14 先草擬幾個token，再核對，能省下什麼？


逐步生成像逐字交稿：寫一個token，等主模型算完，再寫下一個。若能先提出幾個候選，再讓主模型一起核對，便可能減少反覆等待主模型的輪數。但草稿寫得快不夠，還要能放心收下它。

這種做法叫 **speculative decoding，推測解碼**。它分工給草稿方和目標模型：草稿方便宜地提出候選，目標模型決定哪些候選符合原本的生成規則。目標模型就是我們原本要用來回答的模型；草稿方可以是較小模型，也可以是[7.20–7.22的MTP額外分支](https://birdhackor.github.io/tiny-perceptron-vlm/7.20.html)。目的在加速既有生成，不是請較強評審替模型提升答案品質。

先限定最容易核對的 **greedy生成**：每一步都選機率最大的token，並固定並列第一時的選擇規則。假設主模型讀「天氣如何？」後會接今天；讀「天氣如何？／今天」後會接晴天。這裡指定今天、晴天、下雨、嗎各是一個玩具token，不代表實際切詞。

草稿提出「今天／下雨／嗎」。主模型先確認今天相符，再發現第二項應為晴天。它便保留今天，改接晴天，並丟掉下雨之後的嗎：那個嗎是在錯誤的「今天下雨」前文下提出的，不能移到「今天晴天」後照用。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p7-16.14-draft-verify.svg")))

下面用人工規則代替主模型，追蹤保留與修正。字典的key是前文token的tuple，value是該前文下選定的下一token；它不呼叫神經模型，也不量速度。


In [ ]:
rules = {(): "今天", ("今天",): "晴天", ("今天", "晴天"): "〈E〉"}
draft = ["今天", "下雨", "嗎"]
accepted = []
for candidate in draft:
    target_next = rules[tuple(accepted)]
    if candidate != target_next:
        accepted.append(target_next)
        break
    accepted.append(candidate)
    if candidate == "〈E〉":
        break
print("這輪接上的token", accepted)

輸出是`['今天', '晴天']`。第一次核對通過，第二次核對不通過，加入主模型的晴天後就結束本輪；沒有繼續拿錯誤草稿的嗎來核對。接下來若再生成，應從新的正確前文「今天晴天」出發。這樣接受前綴、在第一個不符處修正，會保留原greedy序列。

實際加速的關鍵不在這個Python迴圈。草稿已經給定，主模型可以用因果遮罩，一次向前計算各草稿位置的分布：核對下雨時只讀到今天，核對嗎時才讀到今天下雨。先算出來不代表全部收下；第一個不符後，後面基於錯誤前文的結果仍要丟掉。這與[16.2的逐步decode](https://birdhackor.github.io/tiny-perceptron-vlm/16.2.html)不同，卻不需要讓較早位置偷讀候選的未來。

是否更快，取決於草稿成本、驗證成本與每輪能接受多少項。若草稿常錯，額外工作可能抵銷少跑幾輪的收益；一次核對更多項也會增加運算和快取。MTP提供後續候選，是可以配合的零件，不是「訓練加兩條分支，速度就翻倍」的保證。

練習把草稿改成`["今天", "晴天", "嗎"]`。前兩項通過，第三項應改成〈E〉，輸出是今天、晴天、〈E〉。比較兩份草稿：目標模型的正確序列變了嗎？沒有；改變的是這輪收下多少項，以及在哪裡修正。

<details>
<summary>選讀：隨機抽樣為什麼不能只比第一名？</summary>

本節只展示greedy規則。[Leviathan等人原論文第2.3節與Algorithm 1](https://proceedings.mlr.press/v202/leviathan23a/leviathan23a.pdf)處理隨機抽樣時，還要依目標與草稿機率的比值做接受檢查，拒絕後從校正分布抽樣，才能保持目標分布。只比較兩邊第一名，不能代替這個程序；這裡的目標模型也不是蒸餾時固定提供訓練目標的教師。

[Gloeckle等人的MTP原文第2節與第3.2節](https://arxiv.org/pdf/2404.19737v1)把未來分支用於greedy自我推測解碼；[DeepSeek-V3第2.2節](https://arxiv.org/pdf/2412.19437v2)也允許移除MTP模組、只用主模型正常推論。兩者說明MTP訓練和是否啟用推測解碼，是可以分開決定的事。

</details>
